# Assignment — Missing Values (WKNN) & Data Normalization

**Dataset:** House Data — SIZE (m²), ROOMS, PRICE (million)  
**Tasks:**
1. Use WKNN with k=3 to predict the missing PRICE for NO=7
2. Apply Min-Max, Z-Score, and Decimal Scaling normalization on all features

## Dataset

| NO | SIZE (m²) | ROOMS | PRICE (million) |
|----|-----------|-------|-----------------|
| 1  | 36        | 2     | 200             |
| 2  | 54        | 3     | 350             |
| 3  | 72        | 4     | 450             |
| 4  | 36        | 2     | 250             |
| 5  | 54        | 3     | 300             |
| 6  | 72        | 4     | 500             |
| 7  | 54        | 2     | **?** (missing) |

---
## Part 1 — Missing Values Imputation with WKNN

WKNN (Weighted K-Nearest Neighbors) estimates a missing value using the K most
similar records, where closer neighbors receive higher weights.

Weight formula: $w_i = \frac{1}{d_i^2}$  
Prediction formula: $\hat{y} = \frac{\sum w_i \cdot y_i}{\sum w_i}$

### Step 0 — Load Data

In [1]:
import numpy as np
import pandas as pd

df = pd.DataFrame({
    'NO':    [1,  2,  3,  4,  5,  6,  7],
    'SIZE':  [36, 54, 72, 36, 54, 72, 54],
    'ROOMS': [2,  3,  4,  2,  3,  4,  2],
    'PRICE': [200, 350, 450, 250, 300, 500, np.nan]
})

print("=== Original Dataset ===")
print(df.to_string(index=False))
print(f"\nMissing values:\n{df.isnull().sum()}")

=== Original Dataset ===
 NO  SIZE  ROOMS  PRICE
  1    36      2  200.0
  2    54      3  350.0
  3    72      4  450.0
  4    36      2  250.0
  5    54      3  300.0
  6    72      4  500.0
  7    54      2    NaN

Missing values:
NO       0
SIZE     0
ROOMS    0
PRICE    1
dtype: int64


### Step 1 — Normalize SIZE and ROOMS (Min-Max)

Features must be on the same scale before computing distances.

$$v' = \frac{v - \min}{\max - \min}$$

| Feature | Min | Max | Range |
|---------|-----|-----|-------|
| SIZE    | 36  | 72  | 36    |
| ROOMS   | 2   | 4   | 2     |

In [2]:
def min_max(series):
    return (series - series.min()) / (series.max() - series.min())

df['SIZE_norm']  = min_max(df['SIZE'])
df['ROOMS_norm'] = min_max(df['ROOMS'])

print("=== Normalized Features ===")
print(df[['NO', 'SIZE', 'SIZE_norm', 'ROOMS', 'ROOMS_norm', 'PRICE']].to_string(index=False))

=== Normalized Features ===
 NO  SIZE  SIZE_norm  ROOMS  ROOMS_norm  PRICE
  1    36        0.0      2         0.0  200.0
  2    54        0.5      3         0.5  350.0
  3    72        1.0      4         1.0  450.0
  4    36        0.0      2         0.0  250.0
  5    54        0.5      3         0.5  300.0
  6    72        1.0      4         1.0  500.0
  7    54        0.5      2         0.0    NaN


### Step 2 — Euclidean Distance from NO=7 to Each Training Record

Query point NO=7: SIZE' = 0.500, ROOMS' = 0.000

$$d = \sqrt{(\Delta SIZE')^2 + (\Delta ROOMS')^2}$$

In [3]:
train = df[df['PRICE'].notna()].copy()
query = df.loc[df['NO'] == 7, ['SIZE_norm', 'ROOMS_norm']].values[0]

print(f"Query point (NO=7): SIZE_norm={query[0]:.3f}, ROOMS_norm={query[1]:.3f}")

train['distance'] = np.sqrt(
    (train['SIZE_norm']  - query[0])**2 +
    (train['ROOMS_norm'] - query[1])**2
)

print("\n=== Distance Table ===")
print(train[['NO', 'SIZE_norm', 'ROOMS_norm', 'PRICE', 'distance']]
      .sort_values('distance').to_string(index=False))

Query point (NO=7): SIZE_norm=0.500, ROOMS_norm=0.000

=== Distance Table ===
 NO  SIZE_norm  ROOMS_norm  PRICE  distance
  1        0.0         0.0  200.0  0.500000
  2        0.5         0.5  350.0  0.500000
  4        0.0         0.0  250.0  0.500000
  5        0.5         0.5  300.0  0.500000
  3        1.0         1.0  450.0  1.118034
  6        1.0         1.0  500.0  1.118034


### Step 3 — Select k=3 Nearest Neighbors

In [4]:
k = 3
knn = train.sort_values('distance').head(k).copy()

print(f"=== Top {k} Nearest Neighbors ===")
print(knn[['NO', 'PRICE', 'distance']].to_string(index=False))

=== Top 3 Nearest Neighbors ===
 NO  PRICE  distance
  1  200.0       0.5
  2  350.0       0.5
  4  250.0       0.5


### Step 4 — Compute Weights

$$w_i = \frac{1}{d_i^2}$$

All three neighbors have $d = 0.5$, so $w = 1/0.25 = 4.0$ each.

In [5]:
knn['weight'] = 1 / (knn['distance'] ** 2)

print("=== Neighbors with Weights ===")
print(knn[['NO', 'PRICE', 'distance', 'weight']].to_string(index=False))

=== Neighbors with Weights ===
 NO  PRICE  distance  weight
  1  200.0       0.5     4.0
  2  350.0       0.5     4.0
  4  250.0       0.5     4.0


### Step 5 — Weighted Average → Predicted PRICE

$$\hat{PRICE}_7 = \frac{(4.0 \times 200) + (4.0 \times 350) + (4.0 \times 250)}{4.0+4.0+4.0} = \frac{3200}{12} \approx 266.67$$

In [6]:
predicted_price = (knn['weight'] * knn['PRICE']).sum() / knn['weight'].sum()

print(f"Numerator   = {(knn['weight'] * knn['PRICE']).sum():.2f}")
print(f"Denominator = {knn['weight'].sum():.2f}")
print(f"\n>>> Predicted PRICE for NO=7 = {predicted_price:.2f} million")

df.loc[df['NO'] == 7, 'PRICE'] = round(predicted_price, 2)

print("\n=== Complete Dataset After WKNN Imputation ===")
print(df[['NO', 'SIZE', 'ROOMS', 'PRICE']].to_string(index=False))

Numerator   = 3200.00
Denominator = 12.00

>>> Predicted PRICE for NO=7 = 266.67 million

=== Complete Dataset After WKNN Imputation ===
 NO  SIZE  ROOMS  PRICE
  1    36      2 200.00
  2    54      3 350.00
  3    72      4 450.00
  4    36      2 250.00
  5    54      3 300.00
  6    72      4 500.00
  7    54      2 266.67


---
## Part 2 — Data Normalization

Three normalization methods are applied to the complete dataset (SIZE, ROOMS, PRICE).

### Why Normalize?
House SIZE (36–72 m²) and PRICE (200–500 million) operate on very different scales.
Without normalization, large-scale features unfairly dominate distance calculations
in algorithms like KNN or K-Means — not because they are more important, but simply
because their numeric range is wider.

### Method Formulas

| Method | Formula | Output Range |
|--------|---------|-------------|
| Min-Max | $v' = (v - \min)/(\max - \min)$ | $[0, 1]$ |
| Z-Score | $v' = (v - \bar{A})/\sigma_A$ | Unbounded |
| Decimal Scaling | $v' = v / 10^j$ where $j = \lceil\log_{10}(\max|v|)\rceil$ | $(-1, 1)$ |

### Setup — Prepare Complete Dataset

In [7]:
import math
from sklearn.preprocessing import MinMaxScaler, StandardScaler

features = ['SIZE', 'ROOMS', 'PRICE']
X = df[features].copy()

print("=== Complete Dataset (PRICE for NO=7 = 266.67 from WKNN) ===")
print(df[['NO'] + features].to_string(index=False))

=== Complete Dataset (PRICE for NO=7 = 266.67 from WKNN) ===
 NO  SIZE  ROOMS  PRICE
  1    36      2 200.00
  2    54      3 350.00
  3    72      4 450.00
  4    36      2 250.00
  5    54      3 300.00
  6    72      4 500.00
  7    54      2 266.67


### A. Min-Max Normalization

$$v' = \frac{v - \min(A)}{\max(A) - \min(A)}$$

Implemented using **sklearn's MinMaxScaler**.
Alternatively we also show a custom function to confirm the result.

In [8]:
# --- Using sklearn ---
mm_scaler = MinMaxScaler()
X_mm = pd.DataFrame(
    mm_scaler.fit_transform(X),
    columns=[f + '_mm' for f in features]
)
print("=== A. Min-Max (sklearn) ===")
print(pd.concat([df[['NO']], X_mm], axis=1).round(4).to_string(index=False))

# --- Custom function ---
def min_max_norm(series):
    return (series - series.min()) / (series.max() - series.min())

X_mm_custom = X.apply(min_max_norm)
X_mm_custom.columns = [f + '_mm' for f in features]
print("\n=== A. Min-Max (custom function) ===")
print(pd.concat([df[['NO']], X_mm_custom], axis=1).round(4).to_string(index=False))

=== A. Min-Max (sklearn) ===
 NO  SIZE_mm  ROOMS_mm  PRICE_mm
  1      0.0       0.0    0.0000
  2      0.5       0.5    0.5000
  3      1.0       1.0    0.8333
  4      0.0       0.0    0.1667
  5      0.5       0.5    0.3333
  6      1.0       1.0    1.0000
  7      0.5       0.0    0.2222

=== A. Min-Max (custom function) ===
 NO  SIZE_mm  ROOMS_mm  PRICE_mm
  1      0.0       0.0    0.0000
  2      0.5       0.5    0.5000
  3      1.0       1.0    0.8333
  4      0.0       0.0    0.1667
  5      0.5       0.5    0.3333
  6      1.0       1.0    1.0000
  7      0.5       0.0    0.2222


### B. Z-Score Normalization (Standardization)

$$v' = \frac{v - \bar{A}}{\sigma_A}$$

Implemented using **sklearn's StandardScaler** and also a custom function.

In [9]:
# --- Using sklearn ---
z_scaler = StandardScaler()
X_zs = pd.DataFrame(
    z_scaler.fit_transform(X),
    columns=[f + '_z' for f in features]
)
print("=== B. Z-Score (sklearn) ===")
print(pd.concat([df[['NO']], X_zs], axis=1).round(4).to_string(index=False))

# --- Custom function (population std, ddof=0) ---
def z_score_norm(series):
    return (series - series.mean()) / series.std(ddof=0)

X_zs_custom = X.apply(z_score_norm)
X_zs_custom.columns = [f + '_z' for f in features]
print("\n=== B. Z-Score (custom function) ===")
print(pd.concat([df[['NO']], X_zs_custom], axis=1).round(4).to_string(index=False))

=== B. Z-Score (sklearn) ===
 NO  SIZE_z  ROOMS_z  PRICE_z
  1 -1.3229  -1.0290  -1.2921
  2  0.0000   0.1715   0.1879
  3  1.3229   1.3720   1.1746
  4 -1.3229  -1.0290  -0.7987
  5  0.0000   0.1715  -0.3054
  6  1.3229   1.3720   1.6679
  7  0.0000  -1.0290  -0.6343

=== B. Z-Score (custom function) ===
 NO  SIZE_z  ROOMS_z  PRICE_z
  1 -1.3229  -1.0290  -1.2921
  2  0.0000   0.1715   0.1879
  3  1.3229   1.3720   1.1746
  4 -1.3229  -1.0290  -0.7987
  5  0.0000   0.1715  -0.3054
  6  1.3229   1.3720   1.6679
  7  0.0000  -1.0290  -0.6343


### C. Decimal Scaling Normalization

$$v' = \frac{v}{10^j}, \quad j = \lceil \log_{10}(\max|v|) \rceil$$

**Note:** sklearn does not provide Decimal Scaling — we use a custom function only.

In [10]:
def decimal_scaling(series):
    max_abs = series.abs().max()
    j = math.ceil(math.log10(max_abs))
    return series / (10 ** j), j

X_ds = X.copy().astype(float)
j_values = {}
for col in features:
    X_ds[col], j_values[col] = decimal_scaling(X[col])
X_ds.columns = [f + '_ds' for f in features]

print("=== C. Decimal Scaling (custom function) ===")
print(f"j values => SIZE: {j_values['SIZE']}, ROOMS: {j_values['ROOMS']}, PRICE: {j_values['PRICE']}")
print(pd.concat([df[['NO']], X_ds], axis=1).round(4).to_string(index=False))

=== C. Decimal Scaling (custom function) ===
j values => SIZE: 2, ROOMS: 1, PRICE: 3
 NO  SIZE_ds  ROOMS_ds  PRICE_ds
  1     0.36       0.2    0.2000
  2     0.54       0.3    0.3500
  3     0.72       0.4    0.4500
  4     0.36       0.2    0.2500
  5     0.54       0.3    0.3000
  6     0.72       0.4    0.5000
  7     0.54       0.2    0.2667


### Summary — All Three Methods Combined

In [11]:
summary = pd.concat([
    df[['NO', 'SIZE', 'ROOMS', 'PRICE']],
    X_mm_custom,
    X_zs_custom,
    X_ds
], axis=1)

print("=== Full Normalization Summary ===")
print(summary.round(4).to_string(index=False))
print("\nNote: PRICE for NO=7 uses WKNN imputed value = 266.67 million")

=== Full Normalization Summary ===
 NO  SIZE  ROOMS  PRICE  SIZE_mm  ROOMS_mm  PRICE_mm  SIZE_z  ROOMS_z  PRICE_z  SIZE_ds  ROOMS_ds  PRICE_ds
  1    36      2 200.00      0.0       0.0    0.0000 -1.3229  -1.0290  -1.2921     0.36       0.2    0.2000
  2    54      3 350.00      0.5       0.5    0.5000  0.0000   0.1715   0.1879     0.54       0.3    0.3500
  3    72      4 450.00      1.0       1.0    0.8333  1.3229   1.3720   1.1746     0.72       0.4    0.4500
  4    36      2 250.00      0.0       0.0    0.1667 -1.3229  -1.0290  -0.7987     0.36       0.2    0.2500
  5    54      3 300.00      0.5       0.5    0.3333  0.0000   0.1715  -0.3054     0.54       0.3    0.3000
  6    72      4 500.00      1.0       1.0    1.0000  1.3229   1.3720   1.6679     0.72       0.4    0.5000
  7    54      2 266.67      0.5       0.0    0.2222  0.0000  -1.0290  -0.6343     0.54       0.2    0.2667

Note: PRICE for NO=7 uses WKNN imputed value = 266.67 million
